In [ ]:
import hazm
import pandas as pd
import numpy as np
from sklearn import preprocessing
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
df = pd.read_excel('train_data.xlsx')

In [ ]:
df.head()

In [ ]:
df.info()

The dataset under examination comprises a compilation of Persian sentences alongside the emotional connotations attached to each. It is organized into two distinct columns: one dedicated to the textual content of the sentences and the other to the emotional categorization of these sentences. At the outset, the dataset lacked explicit column titles, necessitating an initial phase of renaming to enhance readability and facilitate easier data manipulation. This renaming process was crucial for streamlining future interactions with the dataset.

Following the assignment of column names, the subsequent task involved scrutinizing the dataset for the presence of missing values or duplicated entries. The inspection yielded reassuring results, confirming the absence of both missing values and duplicates within the dataset. This finding underscores the dataset's integrity and consistency.

To further delve into the characteristics of the dataset, a frequency distribution graph was constructed to visualize the prevalence of different emotional categories among the sentences. The insights derived from this graphical representation indicate that sentiments of happiness predominate among the dataset's entries, whereas expressions of fear are notably less frequent.

# Data cleaning and preprocessing

In [ ]:
normalizer = hazm.Normalizer()

## removing repeated chars like `سلامممممممم`

In [ ]:
df['text'][58]

In [ ]:
df['text'] = df['text'].apply(normalizer.decrease_repeated_chars)

In [ ]:
df['text'][58]

## replace english numbers with persian numbers

In [ ]:
(df['text'][47])

In [ ]:
df['text'] = df['text'].apply(normalizer.persian_number)

In [ ]:
(df['text'][47])

## removing diacritics from words

In [ ]:
df['text'][113]

In [ ]:
df['text'] = df['text'].apply(normalizer.remove_diacritics)

In [ ]:
df['text'][113]

## correcting the spacing in the sentence

In [ ]:
df['text'][204]

In [ ]:
df['text'] = df['text'].apply(normalizer.correct_spacing)

In [ ]:
df['text'][204]

## normalize the text

In [ ]:
df['text'] = df['text'].apply(normalizer.normalize)

## removing stop words

In [ ]:
df['text'][65]

In [ ]:
from hazm import stopwords_list
def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stopwords_list()])

In [ ]:
df['text'] = df['text'].apply(remove_stopwords)

In [ ]:
df['text'][65]

## removing .,?,!,/,...

In [ ]:
import re
def remove_chars(text):
    return re.sub(r'[<>.:()"\'!?؟،,@%$^&*_+\[\]/\\]','',text)

In [ ]:
df['text'][0]

In [ ]:
df['text'] = df['text'].apply(remove_chars)

In [ ]:
df['text'][0]

## Lemmatization

In [ ]:
lemmatizer = hazm.Lemmatizer()

In [ ]:
df['text'] = df['text'].apply(lemmatizer.lemmatize)

In this part, we have performed some preprocessing on the data using the Hazm library, which is designed to process Persian text and data.
The steps include:

1. **Removing Repeated Characters**: The first step addresses instances of excessively repeated characters within sentences, such as "سلامممممممم", by applying a function to decrease the repetition of consecutive identical characters.

2. **Replacing English Numbers with Persian Numbers**: Ensures consistency in numeric representations across the dataset by replacing English numerals with their Persian counterparts.

3. **Removing Diacritics from Words**: Cleanses the text by eliminating diacritical marks from words, enhancing readability and standardizing the text.

4. **Correcting Spacing in Sentences**: Improves sentence structure by adjusting spacing between words to adhere to standard conventions.

5. **Normalizing the Text**: Further refines the text through normalization processes, likely including adjustments for case sensitivity, punctuation, and whitespace.

6. **Removing Stop Words**: Filters out common words that do not contribute significantly to the meaning of sentences, thereby focusing on more relevant content.

7. **Removing Specific Characters**: Cleanses the text by removing a predefined set of characters that are not relevant to the analysis, such as punctuation marks and special symbols.

8. **Lemmatization**: Transforms words into their base or root forms, simplifying the text and reducing complexity.

Each step is meticulously documented with examples before and after the application of the respective function, demonstrating the impact of these preprocessing techniques on the dataset's text data.

# Feature engineering

## Label encoding the target feature

In [ ]:
df.head()

In [ ]:
le = preprocessing.LabelEncoder()
encoded_labels = le.fit_transform(df['mode'])
df['mode_decoded'] = encoded_labels

In [ ]:
df.drop('mode',axis='columns',inplace=True)

In [ ]:
df.head()

## Word tokenization

In [ ]:
tokenizer = hazm.WordTokenizer()

In [ ]:
df['text'] = df['text'].apply(tokenizer.tokenize)

In [ ]:
df.head()

## normalize the tokens

In [ ]:
df['text'] = df['text'].apply(normalizer.token_spacing)

In [ ]:
df.head()

## part 1 word to vector

In [ ]:
wordEmbedding = hazm.WordEmbedding(model_type = 'fasttext')
wordEmbedding.load_model('fasttext_model/fasttext_skipgram_300.bin')

[link](https://mega.nz/file/GqZUlbpS#XRYP5FHbPK2LnLZ8IExrhrw3ZQ-jclNSVCz59uEhrxY) for downloading the `fasttext` model

In [ ]:
wordEmbedding.similarity('پادشاه', 'ملکه')

In [ ]:
wordEmbedding.get_normal_vector('پادشاه')

In [ ]:
def vectorize_sentence(sentence):
    word_vectors = [wordEmbedding.get_normal_vector(word) for word in sentence if word in wordEmbedding.get_vocabs()]
    if len(word_vectors)==0:
        return None
    return sum(word_vectors) / len(word_vectors)

In [ ]:
vectorize_sentence(df['text'][0])

In [ ]:
df['text'][:4].apply(vectorize_sentence)

In [ ]:
df['text_vector'] = df['text'].apply(vectorize_sentence)

In [ ]:
large_array = df['text_vector'][0]
large_array = np.append(large_array , df['mode_decoded'][0]).reshape((1, 301))

In [ ]:
large_array

In [ ]:
temp = df['text_vector'][1]
temp = np.append(temp , df['mode_decoded'][1]).reshape((1, 301))
large_array = np.append(large_array, temp, axis=0)
large_array

In [ ]:
index = 2
for row in df['text_vector'][2:] :
    if row is not None:
        temp = row
        temp = np.append(temp , df['mode_decoded'][index]).reshape((1, 301))
        large_array = np.append(large_array, temp, axis=0)
    index += 1

In [ ]:
large_array

In [ ]:
df2 = pd.DataFrame(large_array)
df2.head()

In [ ]:
df2.describe()

In [ ]:
df2[300].value_counts()

## part 2 TF-IDF

In [ ]:
df.drop(columns='text_vector',inplace=True)

In [ ]:
def embed2string(list):
    return ' '.join(word for word in list)

In [ ]:
df['text'] = df['text'].apply(embed2string)
df

In [ ]:
part1 = df.drop('mode_decoded', axis=1)
part2 = df['mode_decoded']
train_x, test_x, train_y, test_y = train_test_split(
    part1, part2, test_size=0.25, random_state=42)

In [ ]:
train_x

In [ ]:
vectorizer = TfidfVectorizer()
vector = vectorizer.fit_transform(train_x['text'])
train_x = pd.DataFrame(vector.toarray(), columns=vectorizer.get_feature_names_out(), index=train_x.index)

vector2 = vectorizer.transform(test_x['text'])
test_x = pd.DataFrame(vector2.toarray(), columns=vectorizer.get_feature_names_out(), index=test_x.index)

In [ ]:
[train_x.shape,train_y.shape,test_x.shape,test_y.shape]

it was used 2 different type of converting sentence to vector and use them to train to compair them.
The steps include:

### Label Encoding the Target Feature

Initially, the dataset undergoes label encoding of the target feature, 'mode'. This process converts categorical labels into numerical codes, facilitating the use of these features in machine learning algorithms. The encoded labels are stored in a new column, 'mode_decoded', and the original 'mode' column is dropped.

### Word Tokenization

Next, the text data is tokenized using Hazm's `WordTokenizer`. This step breaks down the text into individual words, preparing it for further processing and analysis.

### Normalizing Tokens

The tokens are then normalized to ensure uniformity in spacing between words, improving the consistency of the data.

### Word-to-Vector Conversion

A critical step involves converting words into vectors using Hazm's `WordEmbedding` functionality. This process leverages a pre-trained FastText model to generate dense vector representations of words, capturing semantic meanings. The similarity between two words and their normal vector representation are demonstrated. A custom function, `vectorize_sentence`, aggregates these word vectors into a single vector representing the entire sentence.

### Large Array Construction

The vectors and encoded labels are combined into a large array, `large_array`, which is reshaped and indexed accordingly. This array serves as the basis for creating a new DataFrame, `df2`, showcasing the transformed data.

### TF-IDF Transformation

Finally, the text data undergoes TF-IDF transformation, converting it into a matrix of TF-IDF features. This step involves dropping the 'text_vector' column, converting lists of words back into strings, and splitting the dataset into training and testing sets. The `TfidfVectorizer` from scikit-learn is utilized to transform the text data into TF-IDF vectors, which are then incorporated into the training and testing datasets.

$$ \text{TF} = \frac{\text{Number\: of\, times\: a\: word\: "X"\: appeares\: in\: a\: Document}}{\text{Number\: of\: words\: present\: in\: a\: Document}} $$
$$ \text{IDF} = \log(\frac{\text{Number\: of\, Document\: presents\: in\: a\: Corpus}}{\text{Number\: of\: Documents\: Where\: word\: "X"\: has\: appeared}}) $$
$$ \text{TF-IDF} = \text{TF} \times \text{IDF} $$

Throughout this process, the report emphasizes the importance of each step in preparing the data for machine learning analysis. From encoding categorical variables to transforming text into meaningful numerical representations, each stage contributes to the overall goal of making the data suitable for predictive modeling.

# Train the model using word2vec

## train test split

In [ ]:
part1 = df2.drop(300, axis=1)
part2 = df2[300]
X, X_test, y, y_test = train_test_split(
part1, part2, test_size=0.25, random_state=42)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier,HistGradientBoostingClassifier
from sklearn.ensemble import GradientBoostingClassifier,AdaBoostClassifier,VotingClassifier
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

## finding the best hyperparameter for each model

In [ ]:
def estimate_best_params(model,param_grid,Xtrain,Ytrain):
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5)
    grid_search.fit(Xtrain, Ytrain)
    best_params = grid_search.best_params_
    print(f"best params for {model.__class__.__name__} model is: {best_params}")
    return grid_search.best_estimator_

## Stratified k-fold cv to choose the best model

In [ ]:

skf = StratifiedKFold(n_splits=2)


models = [
    (DecisionTreeClassifier(),
     {'criterion': ['gini', 'entropy'], 'max_depth': [None, 2, 5, 10], 'min_samples_split': [2, 5, 10]}),
    (RandomForestClassifier(),
     {'n_estimators': [100, 200, 300], 'min_samples_split': [2, 3, 4], 'max_depth': [None, 5, 10]}),
    (svm.SVC(), {'kernel': ['linear', 'poly', 'rbf', 'sigmoid'], 'C': [0.1, 1, 10]}),
    (KNeighborsClassifier(), {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance']}),
    (ExtraTreesClassifier(), {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10], 'min_samples_split': [2, 5, 10]}),
    (HistGradientBoostingClassifier(), {}),
    (VotingClassifier(estimators=[('lr', LogisticRegression()), ('svm', svm.SVC(kernel='rbf'))]),
     {}),
    (XGBClassifier(random_state=42),{'max_depth': [5, 10, 15], 'learning_rate': [0.1, 0.2, 0.3]})
]


for model in models:
    best_model = estimate_best_params(model[0],model[1],X,y)
    scores = cross_val_score(best_model, X, y, cv=skf)
    print(f"{best_model.__class__.__name__} Accuracy: {scores.mean():.2f} (+/- {scores.std() * 2:.2f})")

## testing the models on the Test data

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,f1_score

def evaluate(model,Xtrain,Ytrain,Xtest,Ytest):
    # Assume y_test are the true labels and y_pred are the predicted labels
    model.fit(Xtrain,Ytrain)
    y_train_pred = model.predict(Xtrain)
    y_test_pred = model.predict(Xtest)

    print("train report")
    accuracy_train = accuracy_score(Ytrain, y_train_pred)
    print(f"Accuracy: {accuracy_train}")
    weighted_f1_train = f1_score(Ytrain, y_train_pred, average='weighted')
    print(f'Weighted-average F1 Score: {weighted_f1_train}')

    # Confusion Matrix
    cm = confusion_matrix(Ytrain, y_train_pred)
    plt.figure(figsize=(5, 4)) # Set the figure size
    sns.heatmap(cm, annot=True, fmt='d') # Create a heatmap from the confusion matrix
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

    # Precision, Recall, F1-score
    report = classification_report(Ytrain, y_train_pred, output_dict=True)
    colors = ['#ffdfdf', '#dfffff', '#dfffdf', '#dfdfff', '#ffdfff', '#ffffdf']  # Adjust colors as needed

    # Prepare data for plotting
    metrics = ['precision', 'recall', 'f1-score', 'support']
    data = {metric: [] for metric in metrics}
    labels = []

    # Populate data with class-specific metrics
    for cls, metrics_values in report.items():
        if cls.isdigit() or cls in ['macro avg', 'weighted avg']:
            labels.append(cls)
            for metric in metrics:
                data[metric].append(metrics_values.get(metric, None))

    # Convert data to DataFrame
    df3 = pd.DataFrame(data, index=labels)

    # Create subplots for each metric
    fig, axes = plt.subplots(1, len(metrics), figsize=(20, 6), sharey=True)
    for ax, metric in zip(axes, metrics):
        sns.barplot(x=df3.index, y=metric, data=df3, ax=ax, palette=colors)
        ax.set_title(f'{metric.capitalize()} by Class')
        ax.set_ylim(0, 1.1)

    plt.suptitle(f'classification report for {model.__class__.__name__} ', fontsize=16)
    plt.tight_layout()
    plt.show()

    print("\n\n\ntest report")
    accuracy_test = accuracy_score(Ytest, y_test_pred)
    print(f"Accuracy: {accuracy_test}")
    weighted_f1_test = f1_score(Ytest, y_test_pred, average='weighted')
    print(f'Weighted-average F1 Score: {weighted_f1_test}')

    # Confusion Matrix
    cm = confusion_matrix(Ytest, y_test_pred)

    plt.figure(figsize=(5, 4)) # Set the figure size
    sns.heatmap(cm, annot=True, fmt='d') # Create a heatmap from the confusion matrix
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()
    # Precision, Recall, F1-score
    report = classification_report(Ytrain, y_train_pred, output_dict=True)

    colors = ['#ffdfdf', '#dfffff', '#dfffdf', '#dfdfff', '#ffdfff', '#ffffdf']  # Adjust colors as needed

    # Prepare data for plotting
    metrics = ['precision', 'recall', 'f1-score', 'support']
    data = {metric: [] for metric in metrics}
    labels = []

    # Populate data with class-specific metrics
    for cls, metrics_values in report.items():
        if cls.isdigit() or cls in ['macro avg', 'weighted avg']:
            labels.append(cls)
            for metric in metrics:
                data[metric].append(metrics_values.get(metric, None))

    # Convert data to DataFrame
    df3 = pd.DataFrame(data, index=labels)

    # Create subplots for each metric
    fig, axes = plt.subplots(1, len(metrics), figsize=(20, 6), sharey=True)
    for ax, metric in zip(axes, metrics):
        sns.barplot(x=df3.index, y=metric, data=df3, ax=ax, palette=colors)
        ax.set_title(f'{metric.capitalize()} by Class')
        ax.set_ylim(0, 1.1)

    plt.suptitle(f'classification report for {model.__class__.__name__} ', fontsize=16)
    plt.tight_layout()
    plt.show()

    return {"model": model.__class__.__name__, "f1 train": weighted_f1_train, "accuracy train": weighted_f1_train, "f1 test": weighted_f1_test, "accuracy test": weighted_f1_test}

In [ ]:
model_metricx=[]

### Random forest

In [ ]:
model = RandomForestClassifier(max_depth= None, min_samples_split= 4, n_estimators= 200)
model_metricx.append(evaluate(model,X,y,X_test,y_test))

### SVM

In [ ]:
model = svm.SVC(C= 1, kernel='rbf')
model_metricx.append(evaluate(model,X,y,X_test,y_test))

### HistGradientBoostingClassifier

In [ ]:
model = HistGradientBoostingClassifier()
model_metricx.append(evaluate(model,X,y,X_test,y_test))

### VotingClassifier

In [ ]:
model = VotingClassifier(estimators=[('lr', LogisticRegression()), ('svm', svm.SVC(kernel='rbf'))])
model_metricx.append(evaluate(model,X,y,X_test,y_test))

### LogisticRefression

In [ ]:
log_model = LogisticRegression(random_state=42, solver='saga')
model_metricx.append(evaluate(log_model,X,y,X_test,y_test))

### XGBoost

In [ ]:
model = XGBClassifier(random_state=42,learning_rate= 0.3, max_depth= 5)
model_metricx.append(evaluate(model,X,y,X_test,y_test))

### KNN

In [ ]:
model = KNeighborsClassifier( n_neighbors= 7, weights =  'distance')
model_metricx.append(evaluate(model,X,y,X_test,y_test))

In [ ]:
# Convert the list of dictionaries to a DataFrame
df_model = pd.DataFrame(model_metricx)

# Melt the DataFrame to long-form for better visualization
melted_df = df_model.melt(id_vars=["model"], var_name="metric", value_name="value")

# Plotting
plt.figure(figsize=(10, 6))
sns.lineplot(x="model", y="value", hue="metric", data=melted_df)
plt.title('Model Performance Metrics')
plt.xlabel('Models')
plt.ylabel('Performance Metric Value')
plt.xticks(rotation=45)
plt.legend(title='Metric', loc='upper right')
plt.show()

### Model Selection and Hyperparameter Tuning

The initial phase focuses on identifying the best hyperparameters for each model through grid search cross-validation. This systematic approach ensures that each model is optimized for the given dataset, potentially leading to improved performance. Models considered include Decision Trees, Random Forests, Support Vector Machines (SVM), K-Nearest Neighbors (KNN), Extra Trees, Histogram-based Gradient Boosting, AdaBoost, Voting Classifier, Gradient Boosting, and XGBoost classifiers.


### Cross-Validation and Model Evaluation

Stratified k-fold cross-validation is employed to assess the models' performance. This method ensures that each fold of the dataset maintains the original class distribution, providing a fair evaluation of the models. The mean accuracy and standard deviation across folds serve as indicators of the models' reliability and stability.

```
best params for DecisionTreeClassifier model is: {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 2}
DecisionTreeClassifier Accuracy: 0.44 (+/- 0.05)
best params for RandomForestClassifier model is: {'max_depth': None, 'min_samples_split': 4, 'n_estimators': 200}
RandomForestClassifier Accuracy: 0.57 (+/- 0.02)
best params for SVC model is: {'C': 1, 'kernel': 'rbf'}
SVC Accuracy: 0.62 (+/- 0.02)
best params for KNeighborsClassifier model is: {'n_neighbors': 7, 'weights': 'distance'}
KNeighborsClassifier Accuracy: 0.54 (+/- 0.01)
best params for ExtraTreesClassifier model is: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
ExtraTreesClassifier Accuracy: 0.57 (+/- 0.01)
best params for HistGradientBoostingClassifier model is: {}
HistGradientBoostingClassifier Accuracy: 0.60 (+/- 0.01)
best params for VotingClassifier model is: {}
VotingClassifier Accuracy: 0.61 (+/- 0.02)
best params for XGBClassifier model is: {'learning_rate': 0.3, 'max_depth': 5}
XGBClassifier Accuracy: 0.58 (+/- 0.01)
```

### Model Testing on Test Data

After selecting the best models based on cross-validation, the models are evaluated on unseen test data. Performance metrics such as accuracy, precision, recall, F1 score, and confusion matrices are computed to provide a comprehensive assessment of each model's effectiveness. These metrics offer insights into the models' ability to accurately classify instances, their precision in identifying positive instances, and their recall in capturing all relevant instances.

### Individual Model Evaluations

The report concludes with detailed evaluations of selected models, including Random Forest, SVM, HistGradientBoostingClassifier, VotingClassifier, Logistic Regression, and XGBoost. Each model is instantiated with its best-found hyperparameters and evaluated using the training and test datasets. The evaluations include accuracy, F1 score, and visualization of confusion matrices and class-specific performance metrics, providing a clear comparison of the models' capabilities.

# train model for TF-IDF data

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(train_x)
pca = PCA(n_components=2000)
train_x = pca.fit_transform(X_scaled)

X_scaled2 = scaler.transform(test_x)
test_x = pca.transform(X_scaled2)

In [ ]:
model = LogisticRegression(random_state=42, solver='saga')
evaluate(model,train_x,train_y,test_x,test_y)

In [ ]:
model = svm.SVC(C= 1, kernel='rbf')
evaluate(model,train_x,train_y,test_x,test_y)

And this part do the same thing in the previous part but using the tf-ifd data set

# Prediction of test csv

In [ ]:
df_test = pd.read_csv('3rdHW_test.csv')
df_test.columns = ['text']
df_test.head()

In [ ]:
df_test['text'] = df_test['text'].apply(normalizer.decrease_repeated_chars)
df_test['text'] = df_test['text'].apply(normalizer.persian_number)
df_test['text'] = df_test['text'].apply(normalizer.remove_diacritics)
df_test['text'] = df_test['text'].apply(normalizer.correct_spacing)
df_test['text'] = df_test['text'].apply(normalizer.normalize)
df_test['text'] = df_test['text'].apply(remove_stopwords)
df_test['text'] = df_test['text'].apply(remove_chars)
df_test['text'] = df_test['text'].apply(lemmatizer.lemmatize)
df_test['text'] = df_test['text'].apply(tokenizer.tokenize)
df_test['text'] = df_test['text'].apply(normalizer.token_spacing)
df_test['text_vector'] = df_test['text'].apply(vectorize_sentence)
large_array = df_test['text_vector'][0].reshape((1, 300))
temp = df_test['text_vector'][1].reshape((1, 300))
large_array = np.append(large_array, temp, axis=0)
index = 2
for row in df_test['text_vector'][2:]:
    if row is not None:
        temp = row.reshape((1, 300))
        large_array = np.append(large_array, temp, axis=0)
    index += 1
df_test2 = pd.DataFrame(large_array)
df_test2.head()

In [ ]:
svm_model = svm.SVC(C= 1, kernel='rbf')
svm_model.fit(X,y)
predictions = svm_model.predict(df_test2)
predictions = np.append(predictions,2)

In [ ]:
def manual_inverse_transform(encoded_labels, le):
    encoded_labels = list(encoded_labels)
    original_labels=[le.classes_[int(label)] for label in encoded_labels]
    return original_labels

In [ ]:
original_predictions = manual_inverse_transform(predictions,le)

In [ ]:
res = pd.read_csv('3rdHW_test.csv')
res['preds'] = original_predictions
res.columns = ['X','Y']
res

In [ ]:
res.to_csv('result.csv', index=False)